In [7]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [10]:
import sys
sys.path.insert(0, '/app/04-evaluation/code')

In [11]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [12]:
for q in ground_truth:
    #q = ground_truth[g]
    print(q)


{'question': 'Is it okay to join the course late if I just found it now?', 'document': '74eb249bbf'}
{'question': 'Can I still take this course even if I missed the start date?', 'document': '74eb249bbf'}
{'question': 'If I join after the course has already started, am I still eligible for a certificate?', 'document': '74eb249bbf'}
{'question': 'Do I need to submit my project before submissions close to get the certificate?', 'document': '74eb249bbf'}
{'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?', 'document': '74eb249bbf'}
{'question': 'I registered for the LLM Zoomcamp — when should I expect a confirmation email?', 'document': '977bf7786c'}
{'question': 'Do I actually need an acceptance email before I can start the course and hand in homework?', 'document': '977bf7786c'}
{'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?', 'document': '977bf7786c'}
{'question': 'Is regis

In [13]:
#q=ground_truth[10]

In [14]:
doc_idx = {}
for doc in documents:
    doc_idx[doc["doc_id"]]=doc

In [15]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()


#openai_client = OpenAI()    

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
        timeout=15.0,
    max_retries=1,
)

In [16]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
    )

In [17]:
q['question']

'How do I fix the requests version mismatch when setting up the dlt workshop environment?'

In [18]:
answer = assistant.rag(q['question'])
print(answer)

Based on the provided context, the question is:

"How do I fix the requests version mismatch when setting up the dlt workshop environment?"

According to the context, you can fix the requests version mismatch by using the following command:

```
pip install "requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip"
```

This command installs the requests library directly from the source, specifically the version 2.32.3, which is the required version for lancedb.


In [19]:
doc_idx[q['document']]

{'course': 'llm-zoomcamp',
 'section': 'Workshop: Open-Source Data Ingestion (dlt)',
 'question': 'Error: How to fix requests library only installs v2.28 instead of v2.32 required for lancedb?',
 'answer': 'If you encounter a 401 Client Error, it may indicate the need to grant access to the key or that the key is incorrect.\n\nTo install the correct version directly from the source, use the following command:\n\n```bash\npip install "requests @ https://github.com/psf/requests/archive/refs/tags/v2.32.3.zip"\n```',
 'doc_id': '4b30b918bc'}

In [20]:
### Runing RAG for one question

rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
print(answer_llm)

According to the course-related information, it is okay to join the course late if you just found it now. The course does not have any specific time restrictions for joining, and you can still submit your project to receive a certificate while submissions are still accepted.


In [21]:
assistant.total_cost()

0.00148875

In [22]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

print(answer_orig)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [23]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

print(rag_result)

{'question': 'Is it okay to join the course late if I just found it now?', 'answer_llm': 'According to the course-related information, it is okay to join the course late if you just found it now. The course does not have any specific time restrictions for joining, and you can still submit your project to receive a certificate while submissions are still accepted.', 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'document': '74eb249bbf'}


### Processing all questions


In [24]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [25]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Based on the context, to your question: "Yes, it is okay to join the course late if I just found it now." If you want to receive a certificate, you\'ll need to submit your project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
assistant.total_cost()

0.0020805

In [27]:
assistant.reset_usage()

In [28]:
assistant.total_cost()

0.0

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, ground_truth[:6], generate_rag_answer)

  0%|          | 0/6 [00:00<?, ?it/s]

In [31]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [32]:
assistant.total_cost()

0.00504375

In [34]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new-v2.csv", index=False)